# Video 10: Production CrewAI Agentic AI

## What We Will Build
In this notebook, we will build an **End-to-End CrewAI Production Project** that combines everything:
- Supervisor/manager workflow
- Multiple specialized agents (Researcher, Analyst, Report Writer)
- Custom tools for real-world capabilities
- RAG with vector database for knowledge
- Memory for conversation persistence
- Validation and review agents
- Error handling and guardrails
- Logging and observability
- Streamlit interface for user interaction

This is the capstone project that brings together all CrewAI concepts.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai langchain-community faiss-cpu python-dotenv streamlit

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom crewai.tools import toolfrom langchain_openai import ChatOpenAI, OpenAIEmbeddingsfrom langchain_community.vectorstores import FAISSfrom langchain_core.documents import Documentimport osimport jsonimport loggingfrom datetime import datetimefrom dotenv import load_dotenv# Setup logginglogging.basicConfig(    level=logging.INFO,    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')logger = logging.getLogger('ProductionCrew')# Load environment variablesload_dotenv()# Validate API keyapi_key = os.getenv('OPENAI_API_KEY')if not api_key:    raise ValueError('OPENAI_API_KEY not found in environment. Check your .env file.')# Initialize LLM and Embeddingsllm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=api_key)embeddings = OpenAIEmbeddings(api_key=api_key)logger.info('Production environment initialized successfully')print('Production environment initialized successfully')

## 3. Theory: Production Architecture

### Production Considerations
| Aspect | Development | Production |
|--------|-------------|------------|
| LLM | gpt-4o-mini | gpt-4o or custom model |
| Memory | MemorySaver | Postgres/Redis |
| Vector DB | FAISS (local) | Pinecone/Weaviate |
| Logging | print() | Structured logging |
| Error Handling | Basic | Comprehensive + fallback |
| Guardrails | Minimal | Input/output validation |
| Monitoring | Manual | LangSmith/Datadog |
| Deployment | Local | Docker/Kubernetes |

### Architecture Overview
```
User Input
    |
    ▼
[Router Agent] ──> [Research Agent] ──> [Analysis Agent] ──> [Report Agent]
    |                    |                      |                    |
    |               [RAG Tool]           [Calc Tool]         [Review Agent]
    |                    |                      |                    |
    └────────────────────┴──────────────────────┴────────────────────┘
                              |
                              ▼
                        [Final Output]
```

## 4. Setup RAG Knowledge Base

In [ ]:
# Production-grade knowledge baseknowledge_documents = [    Document(        page_content=(            'CrewAI is a cutting-edge framework for orchestrating multi-agent AI systems. '            'It enables the creation of role-based agents that collaborate on complex tasks. '            'Key features include sequential and hierarchical processes, built-in memory, '            'custom tools, and agent delegation. CrewAI is built on top of LangChain.'        ),        metadata={'source': 'docs', 'category': 'framework', 'version': '1.0'}    ),    Document(        page_content=(            'LangChain is a framework for developing applications powered by language models. '            'It provides tools, chains, agents, and integrations with vector stores, APIs, and databases. '            'LangChain supports multiple LLM providers including OpenAI, Anthropic, and local models.'        ),        metadata={'source': 'docs', 'category': 'framework', 'version': '1.0'}    ),    Document(        page_content=(            'Vector databases store high-dimensional embeddings for semantic search. '            'Popular options include FAISS (local), Pinecone (cloud), Weaviate, and ChromaDB. '            'They enable Retrieval-Augmented Generation (RAG) by finding semantically similar documents.'        ),        metadata={'source': 'docs', 'category': 'database', 'version': '1.0'}    ),    Document(        page_content=(            'Agentic AI refers to AI systems that can autonomously perceive, decide, and act to achieve goals. '            'Unlike traditional AI, agentic systems can use tools, plan multi-step strategies, and adapt to '            'changing environments. CrewAI and LangGraph are leading frameworks for building agentic AI.'        ),        metadata={'source': 'docs', 'category': 'concept', 'version': '1.0'}    ),]# Create vector storevector_store = FAISS.from_documents(knowledge_documents, embeddings)logger.info(f'Knowledge base created with {len(knowledge_documents)} documents')print(f'Knowledge base created with {len(knowledge_documents)} documents')

## 5. Define Production Tools

In [ ]:
@tooldef search_knowledge_base(query: str) -> str:    """Search the knowledge base for relevant information.        Args:        query: The search query to find relevant documents    """    try:        docs = vector_store.similarity_search(query, k=2)        if not docs:            return 'No relevant documents found in the knowledge base.'                results = []        for i, doc in enumerate(docs, 1):            results.append(f"Source {i} [{doc.metadata['category']}]: {doc.page_content}")        return '\n\n'.join(results)    except Exception as e:        logger.error(f'Knowledge base search error: {e}')        return f'Error searching knowledge base: {str(e)}'@tooldef calculate(expression: str) -> str:    """Perform safe mathematical calculations.        Args:        expression: Mathematical expression like '2 + 2 * 5'    """    try:        allowed = {'abs': abs, 'round': round, 'max': max, 'min': min, 'sum': sum, 'pow': pow}        result = eval(expression, {'__builtins__': {}}, allowed)        return f'Calculation result: {result}'    except Exception as e:        logger.error(f'Calculation error: {e}')        return f'Calculation error: {str(e)}'@tooldef get_current_date() -> str:    """Get the current date and time."""    return f'Current date and time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'production_tools = [search_knowledge_base, calculate, get_current_date]logger.info(f'{len(production_tools)} production tools initialized')print(f'{len(production_tools)} production tools initialized')

## 6. Create Production Agents

In [ ]:
# Router/Supervisor Agentsupervisor = Agent(    role='Project Supervisor',    goal='Coordinate the team, assign tasks, and ensure high-quality deliverables',    backstory=(        'You are an experienced project supervisor who excels at understanding requirements, '        'delegating tasks to the right specialists, and reviewing work for quality. '        'You ensure all outputs meet professional standards.'    ),    llm=llm,    verbose=True,    allow_delegation=True,    max_iter=5)# Research Agentresearcher = Agent(    role='Senior Research Specialist',    goal='Gather comprehensive, accurate, and up-to-date information on any topic',    backstory=(        'You are a world-class researcher with expertise across technology, business, and science. '        'You use knowledge base tools to ground your findings and always cite your sources. '        'You are thorough, accurate, and detail-oriented.'    ),    llm=llm,    tools=[search_knowledge_base, get_current_date],    verbose=True,    allow_delegation=True,    max_iter=5)# Analysis Agentanalyst = Agent(    role='Strategic Data Analyst',    goal='Transform raw data into actionable strategic insights and recommendations',    backstory=(        'You are a senior data analyst with 10 years of experience in strategic consulting. '        'You identify patterns, trends, and opportunities that others miss. '        'You use calculation tools to verify your analysis and provide data-driven recommendations.'    ),    llm=llm,    tools=[calculate, get_current_date],    verbose=True,    allow_delegation=True,    max_iter=5)# Report Writer Agentreport_writer = Agent(    role='Professional Report Writer',    goal='Create polished, well-structured, and engaging reports for executive audiences',    backstory=(        'You are an award-winning report writer who crafts executive summaries, industry reports, '        'and strategic documents. Your writing is clear, concise, and impactful. '        'You adapt your tone and style for different audiences.'    ),    llm=llm,    verbose=True,    allow_delegation=False,    max_iter=5)# Review Agentreviewer = Agent(    role='Quality Assurance Reviewer',    goal='Ensure all deliverables are accurate, complete, and meet quality standards',    backstory=(        'You are a meticulous quality assurance specialist who reviews content for accuracy, '        'completeness, clarity, and professionalism. You provide constructive feedback '        'and quality scores.'    ),    llm=llm,    verbose=True,    allow_delegation=False,    max_iter=3)logger.info('All production agents created')print('All production agents created successfully')

## 7. Define Production Tasks

In [ ]:
# Task 1: Researchresearch_task = Task(    description=(        'Research the topic: {topic}. Use the search_knowledge_base tool to find relevant information. '        'Provide a comprehensive research report with specific data points, examples, and sources. '        'Include the current date in your findings.'    ),    expected_output=(        'A comprehensive research report (minimum 400 words) with clear sections, '        'specific data points, and source citations.'    ),    agent=researcher)# Task 2: Analysisanalysis_task = Task(    description=(        'Analyze the research findings on: {topic}. Identify key trends, patterns, '        'opportunities, and risks. Use calculation tools where appropriate. '        'Provide strategic recommendations with data-backed justification.'    ),    expected_output=(        'A strategic analysis report (minimum 300 words) with trends, insights, '        'and actionable recommendations.'    ),    agent=analyst,    context=[research_task])# Task 3: Report Writingreport_task = Task(    description=(        'Write a professional executive report on: {topic}. Use the research and analysis '        'provided. Include: executive summary, background, key findings, analysis, '        'recommendations, and conclusion. Format for executive audience.'    ),    expected_output=(        'A polished executive report (minimum 500 words) with professional formatting '        'and clear sections.'    ),    agent=report_writer,    context=[research_task, analysis_task])# Task 4: Quality Reviewreview_task = Task(    description=(        'Review the executive report on: {topic}. Check for: 1) Factual accuracy, '        '2) Completeness of all sections, 3) Clarity and readability, 4) Professional tone, '        '5) Actionability of recommendations. Provide a quality score (1-10) and specific feedback.'    ),    expected_output=(        'A detailed review with quality score, strengths, weaknesses, and improvement suggestions.'    ),    agent=reviewer,    context=[report_task])logger.info('All production tasks defined')print('All production tasks defined successfully')

## 8. Build the Production Crew

In [ ]:
production_crew = Crew(    agents=[supervisor, researcher, analyst, report_writer, reviewer],    tasks=[research_task, analysis_task, report_task, review_task],    process=Process.hierarchical,    manager_agent=supervisor,    verbose=True,    memory=True,    max_rpm=20)logger.info('Production crew compiled')print('Production crew compiled successfully')print(f'  Process: {production_crew.process}')print(f'  Agents: {len(production_crew.agents)}')print(f'  Tasks: {len(production_crew.tasks)}')print(f'  Memory: Enabled')print(f'  Max RPM: {production_crew.max_rpm}')

## 9. Execute the Production Crew

In [ ]:
# Define the topictopic = 'The Future of Agentic AI in Enterprise Software'logger.info(f'Starting production crew execution for topic: {topic}')print('\n' + '=' * 70)print('EXECUTING PRODUCTION CREW')print('=' * 70)print(f'Topic: {topic}')print('=' * 70)# Execute with inputsresult = production_crew.kickoff(inputs={'topic': topic})logger.info('Production crew execution complete')print('\n' + '=' * 70)print('EXECUTION COMPLETE')print('=' * 70)

## 10. Display and Save Results

In [ ]:
# Display final outputprint('\n' + '=' * 70)print('FINAL OUTPUT')print('=' * 70)print(result)# Save to fileoutput_file = 'production_report.txt'with open(output_file, 'w', encoding='utf-8') as f:    f.write(f'Production Crew Report\n')    f.write(f'Topic: {topic}\n')    f.write(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')    f.write('=' * 70 + '\n\n')    f.write(str(result))logger.info(f'Report saved to {output_file}')print(f'\nReport saved to: {output_file}')

## 11. Production Checklist

- [x] **Multi-agent system** with supervisor, researcher, analyst, writer, reviewer
- [x] **Custom tools** for knowledge search, calculations, and date retrieval
- [x] **RAG** with FAISS vector database for grounded responses
- [x] **Memory** enabled for context continuity
- [x] **Hierarchical process** with manager oversight
- [x] **Validation** via quality assurance reviewer
- [x] **Error handling** in tools with logging
- [x] **Structured logging** for observability
- [x] **Input validation** via expected_output definitions
- [x] **Output persistence** to file

### Deployment Steps
1. Create `.env` file with API keys
2. Run `pip install -r requirements.txt`
3. Execute notebook or run as script
4. For web UI, use the Streamlit app below

## 12. Streamlit Interface (Bonus)

Save this as `app.py` in the same directory and run with `streamlit run app.py`:

```python
import streamlit as st
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

st.set_page_config(page_title='Production CrewAI', page_icon='🤖')
st.title('🤖 Production CrewAI Agentic AI')
st.write('Powered by CrewAI with RAG, Tools, Memory, and Multi-Agent Collaboration')

# Sidebar
st.sidebar.header('Configuration')
topic = st.sidebar.text_input('Research Topic', 'The Future of AI in Healthcare')
process_type = st.sidebar.selectbox('Process Type', ['Hierarchical', 'Sequential'])
memory_enabled = st.sidebar.checkbox('Enable Memory', value=True)

# Main
if st.button('🚀 Run Crew'):
    with st.spinner('Running multi-agent crew...'):
        # Initialize LLM
        llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv('OPENAI_API_KEY'))
        
        # Create agents (simplified for demo)
        researcher = Agent(role='Researcher', goal=f'Research {topic}', backstory='Expert researcher.', llm=llm, verbose=False)
        writer = Agent(role='Writer', goal=f'Write about {topic}', backstory='Expert writer.', llm=llm, verbose=False)
        
        task1 = Task(description=f'Research {topic}', expected_output='Research report.', agent=researcher)
        task2 = Task(description=f'Write report on {topic}', expected_output='Final report.', agent=writer, context=[task1])
        
        process = Process.hierarchical if process_type == 'Hierarchical' else Process.sequential
        crew = Crew(agents=[researcher, writer], tasks=[task1, task2], process=process, memory=memory_enabled, verbose=False)
        
        result = crew.kickoff(inputs={'topic': topic})
    
    st.success('Crew execution complete!')
    st.markdown('### Final Output')
    st.markdown(str(result))
```

## 13. Guardrails and Security

```python
# Input validation
def validate_input(topic: str) -> bool:
    if not topic or len(topic) < 3:
        return False
    if any(bad in topic.lower() for bad in ['injection', 'ignore previous']):
        return False
    return True

# Output filtering
def filter_output(text: str) -> str:
    # Remove PII, harmful content, etc.
    return text

# Rate limiting
from time import sleep
sleep(1)  # Between API calls
```

---

## ✅ Summary

You have successfully built a **Production-Grade CrewAI Agentic AI System** with:
1. ✅ Supervisor workflow with 5 specialized agents
2. ✅ Custom tools (knowledge search, calculator, date)
3. ✅ RAG with FAISS vector database
4. ✅ Memory enabled for context continuity
5. ✅ Hierarchical process with manager oversight
6. ✅ Validation via quality assurance reviewer
7. ✅ Comprehensive error handling and logging
8. ✅ Output persistence and Streamlit UI
9. ✅ Production deployment checklist

**Congratulations! You have completed the CrewAI course! 🎉**